In [ ]:
from probabilistic_word_embeddings.estimation import map_estimate 
from probabilistic_word_embeddings.embeddings import Embedding
from simulate import  load_simulated_data_generator, load
import tensorflow as tf

import numpy as np

def sigmoid(x:float): # might be a good idea to have this in a common library.
  return 1 / (1 + np.exp(-x))


def sufficient_statistics(data, vocabulary):
    V = len(vocabulary)
    count_matrix_x1 = np.zeros((V, V), dtype=int)
    count_matrix_total = np.zeros((V, V), dtype=int)

    for item in data:
        v_index = vocabulary[item['v']]
        w_index = vocabulary[item['w']]
        if item['x'] == 1:
            count_matrix_x1[v_index, w_index] += 1
        #elif item['x'] == 0:
        #    count_matrix_x0[v_index, w_index] += 1
        count_matrix_total[v_index, w_index] += 1

    return count_matrix_x1, count_matrix_total

def calculate_p(vi, wi, theta, V):
    rho_v = theta[vi, :]          # $\rho_v = \theta_v$
    alpha_w = theta[wi + V, :]    # $\alpha_w = \theta_{V + w}$
    eta = rho_v.dot(alpha_w)      # $\eta = \rho_v^T \alpha_w$
    p = sigmoid(eta)              # Run it through the link function $\sigma; p = \sigma(\eta)$
    return p

def compare_theta_to_sufficent_statistics(v, w, theta, counts_1, counts_total, vocabulary):
    V=  len(vocabulary)

    if v == w:
        return None, None
    vi, wi = vocabulary[v], vocabulary[w]
    
    
    rho_v = theta[vi, :]          # $\rho_v = \theta_v$
    alpha_w = theta[wi + V, :]    # $\alpha_w = \theta_{V + w}$
    eta = rho_v.dot(alpha_w)      # $\eta = \rho_v^T \alpha_w$
    p = sigmoid(eta)              # Run it through the link function $\sigma; p = \sigma(\eta)$


    observed_ratio = counts_1[vi, wi] / counts_total[vi, wi] if counts_total[vi, wi] > 0 else 0.0
    return (p, observed_ratio)


def compare_thetas(v, w, theta1, theta2, vocabulary):
    V=  len(vocabulary)

    if v == w:
        return None, None
    vi, wi = vocabulary[v], vocabulary[w]
    
    
    p1 = calculate_p(vi, wi, theta1, V)
    p2 = calculate_p(vi, wi, theta2, V)

    return (p1, p2)

In [ ]:

import os

for i in os.listdir():
    if i.split(".")[-1] == 'json':
        print(i)

In [ ]:


file_path = 'test100k_d2_s08.json'
for item in load_simulated_data_generator(file_path, batch_size=5):
    print(item)  
    break

In [ ]:
json_ = load(file_path)
json_['theta']

In [ ]:
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

display(counts_1)

display(counts_total)

In [ ]:
vocabulary = json_['vocabulary']
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

words = list(vocabulary.keys())
theta = np.array(json_['theta'])

store_p = []
store_observer_ratios = []
for v in words:
    for w in words:
        p, observed_ratio = compare_theta_to_sufficent_statistics(v, w, theta, counts_1, counts_total, vocabulary)

        if p is None:
            pass
        else:
            print(f'{v}, {w}')
            print('from theta :', p)
            print('observed   :', observed_ratio)
            print()
            store_p.append(p)
            store_observer_ratios.append(observed_ratio)


print('CORRELATION:', np.corrcoef(store_p, store_observer_ratios)[1,0])


In [ ]:

N = 100000
model = 'sgns'
# ln(2) = 0.6931471805599453
#python simulate.py --save_path "test100k.json" --vocab_size 10 --dimensionality 3 --n_datapoints 100000 --seed 1 --sd 0.5
e = Embedding(vocabulary=set(json_['vocabulary'].keys()), dimensionality=2, lambda0=0.0)


theta_estimated_before = e[list(vocabulary.keys()) + [w + '_c' for w in list(vocabulary.keys())]].numpy()
magnitude_before =  tf.reduce_mean(tf.math.abs(theta_estimated_before))
print('Record magnitude before:' , magnitude_before)

estimate = map_estimate(e, data_generator=load_simulated_data_generator(file_path, batch_size=100), batch_size=100, N = N, evaluate=False, training_loss=True, epochs = 30, model=model)

In [ ]:
vocabulary = json_['vocabulary']
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

words = list(vocabulary.keys())
#theta = np.array(json_['theta'])

theta_estimated = e[list(vocabulary.keys()) + [w + '_c' for w in list(vocabulary.keys())]].numpy()

store_p = []
store_p_estimated = []
for v in words:
    for w in words:
        p, p_estimated = compare_thetas(v, w, theta, theta_estimated, vocabulary)

        if p is None:
            pass
        else:
            print(f'{v}, {w}')
            print('from theta :', p)
            print('observed   :', p_estimated)
            print()
            store_p.append(p)
            store_p_estimated.append(p_estimated)


print('CORRELATION:', np.corrcoef(store_p, store_p_estimated)[1,0])


In [ ]:
print('Magnitude before:',  tf.reduce_mean(tf.math.abs(theta_estimated_before)))
theta_estimated = e[list(vocabulary.keys()) + [w + '_c' for w in list(vocabulary.keys())]].numpy()
print('Magnitude after :', tf.reduce_mean(tf.math.abs(theta_estimated)))




In [ ]:
tf.__version__